# 00 — Quadruped-PyMPCの背景・目的・結論と学習地図

## 1. このリポジトリはなぜ作られたか

四足歩行では、目標速度を与えるだけでは脚は動きません。「いつ接地するか」「次にどこへ着地するか」「各接地点で何Nの床反力を出すか」「その力をどの関節トルクで作るか」を、転倒しない速さで繰り返し決める必要があります。

`iit-DLSLab/Quadruped-PyMPC` は、この問題を **Single Rigid Body Dynamics（SRBD）に基づくModel Predictive Control** として解く研究・実装リポジトリです。標準経路はCasADiで非線形モデルを記述し、acadosで有限時間最適制御問題を反復的に解きます。MuJoCo上の複数四足ロボットに加え、公開READMEではUnitree系実機との接続も想定されています。

## 2. 目的

この教材の目的はAPIの使い方だけを覚えることではありません。現行コードを正本として、

1. 速度指令から接地列・着地点参照が作られる理由
2. SRBDの式がCasADi変数へどう対応するか
3. Q/R、摩擦円錐、接触制約がacados OCPへどう入るか
4. 最適床反力がJacobian転置で関節トルクへ変換される理由
5. 症状からチューニング箇所を選び、式をテスト付きで変更する方法

を、処理ブロックごとに理解することです。

## 3. 先に結論

このリポジトリの強みは、**Gait → foothold → centroidal NMPC → stance/swing torque → MuJoCo** がPythonで一気通貫し、gradient-based MPCとsampling-based MPCの研究分岐も同じ枠組みに置かれていることです。一方、標準nominalモデルは脚の全身力学を予測するモデルではなくSRBD近似です。性能は地形、速度、摩擦モデル、gait、トルク飽和、solver周期の組合せに依存し、READMEの機能一覧だけでは判断できません。そのため最後に実コードを30シナリオで計測します。

**前提**: なし

> 各章は「背景 → ASCIIデータ流 → 数式 → コメント付きコード → 実行結果 → 限界」の順に読みます。`実装事実` と`学習用近似`を混同しません。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 4. 1制御周期のASCIIデータフロー

```text
┌─────────────────────────────────────────────────────────────────────────────┐
│ User / command generator                                                   │
│ 目標: v_ref(W) [3] m/s, omega_ref [3] rad/s                               │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ MuJoCo / gym-quadruped plant                                                │
│ 観測: COM, base姿勢・速度, 足位置・速度, q, qdot, J, M, contact             │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ WBInterface — reference generation                                         │
│  ├─ PeriodicGaitGenerator: phase φ_i → contact c_i,k ∈ {0,1} [4×N]         │
│  ├─ FootholdReferenceGenerator: p_hip, v, v_ref → p_foot_ref                │
│  └─ SwingTrajectoryController: lift-off → spline → p, pdot, pddot           │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ SRBDControllerInterface                                                    │
│ state/ref/contact/inertia を nominal・input_rates・sampling等へ振り分ける   │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ Nominal centroidal NMPC (CasADi + acados)                                  │
│ min Σ||x-x_ref||²_Q + ||u-u_ref||²_R                                       │
│ s.t. SRBD, contact mask, friction cone, GRF/foothold constraints            │
│ 出力: 第0予測段の GRF F_i [12] N と optimized foothold [4×3] m             │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ Stance / swing torque                                                       │
│ stance: tau_i = -J_i^T F_i   swing: Cartesian PD + feedback linearization  │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ actuator orderへ格納 → 90% soft torque clip → env.step(action)             │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               └──────── 次の観測へ戻る ───────────────────────┘
```

式で縮めると、

\[
v^{ref}\rightarrow (c_{i,k},p^{ref}_{foot})
\rightarrow \operatorname{NMPC}(x,u)
\rightarrow F_i^{cmd}\rightarrow \tau_i\rightarrow x_{next}
\]

です。標準`nominal` MPCが直接最適化する入力は、足速度12要素と床反力12要素です。関節トルクは後段で計算されます。

In [2]:
learning_order = [
    "01 environment/source map",
    "02 vectors, units, frames",
    "03 MuJoCo plant",
    "04 closed-loop timing",
    "05 gait/contact schedule",
    "06 foothold reference",
    "07 swing trajectory",
    "08 SRBD dynamics",
    "09 friction/contact constraints",
    "10 MPC objective",
    "11 receding horizon/acados",
    "12 force-to-torque conversion",
    "13 end-to-end baseline",
    "14 logging and diagnosis",
    "15 tuning laboratory",
    "16 equation modification capstone",
]
for i, item in enumerate(learning_order, 1):
    print(f"{i:02d}. {item}")

01. 01 environment/source map
02. 02 vectors, units, frames
03. 03 MuJoCo plant
04. 04 closed-loop timing
05. 05 gait/contact schedule
06. 06 foothold reference
07. 07 swing trajectory
08. 08 SRBD dynamics
09. 09 friction/contact constraints
10. 10 MPC objective
11. 11 receding horizon/acados
12. 12 force-to-torque conversion
13. 13 end-to-end baseline
14. 14 logging and diagnosis
15. 15 tuning laboratory
16. 16 equation modification capstone


## 進級条件

- **理解**: shape・単位・frameを添えて信号を説明できる
- **再現**: Notebookを上から再実行して同じ結論になる
- **調整**: 一度に1群だけ変更し、仮説と評価量を先に書く
- **改造**: 式、CasADi式、制約、テストを同じ変更単位で扱う

`14` までは上流コードを変更しません。`15` は設定値の比較、`16` はNotebook内で
式の候補を検証します。上流ファイルの変更は、比較テストができてからです。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。